I want to first load my JSON file and check what type it is, how many entries, and what the first entry looks like

In [4]:
import json
import pandas as pd

with open("likes_and_reactions.json", "r", encoding="utf-8") as file:
    facebook_data = json.load(file)

type(facebook_data)

list

In [5]:
len(facebook_data)

10756

In [6]:
facebook_data[0]

{'timestamp': 1788134625,
 'media': [],
 'label_values': [{'label': 'Reaction', 'value': 'Like'},
  {'label': 'URL',
   'value': 'https://www.facebook.com/christine.s.aiken/posts/pfbid02hjVjMADYjon9dHgq3MWTxUyWnmn7uYiNDcmG6ErzC1Y2cPMx9Xx5sGnGQ6CDCoj2l',
   'href': 'https://www.facebook.com/christine.s.aiken/posts/pfbid02hjVjMADYjon9dHgq3MWTxUyWnmn7uYiNDcmG6ErzC1Y2cPMx9Xx5sGnGQ6CDCoj2l'},
  {'label': 'Title', 'value': ''},
  {'label': 'Name', 'value': 'Christine Giver'}],
 'fbid': '10164458003235399'}

In [7]:
facebook_data[0].keys()

dict_keys(['timestamp', 'media', 'label_values', 'fbid'])

Label values is what matters here, but they are nested inside something else

In [10]:
facebook_data[0]["label_values"][0]

{'label': 'Reaction', 'value': 'Like'}

In [11]:
facebook_data[0]["label_values"][0]["value"]

'Like'

In [12]:
facebook_data[0]["label_values"][1]

{'label': 'URL',
 'value': 'https://www.facebook.com/christine.s.aiken/posts/pfbid02hjVjMADYjon9dHgq3MWTxUyWnmn7uYiNDcmG6ErzC1Y2cPMx9Xx5sGnGQ6CDCoj2l',
 'href': 'https://www.facebook.com/christine.s.aiken/posts/pfbid02hjVjMADYjon9dHgq3MWTxUyWnmn7uYiNDcmG6ErzC1Y2cPMx9Xx5sGnGQ6CDCoj2l'}

In [13]:
facebook_data[0]["label_values"][2]

{'label': 'Title', 'value': ''}

In [14]:
facebook_data[0]["label_values"][3]

{'label': 'Name', 'value': 'Christine Giver'}

0. reaction
1. URL
2. Title
3. Name

In [15]:
facebook_data[0]["label_values"][3]["value"]

'Christine Giver'

This unfortunately won't always work because they might be in different locations

In [16]:
facebook_data[0]["label_values"]
facebook_data[1]["label_values"]
facebook_data[2]["label_values"]
facebook_data[10]["label_values"]

[{'label': 'Reaction', 'value': 'Like'},
 {'label': 'URL',
  'value': 'https://www.facebook.com/sharon.merklin/posts/pfbid0vGqUJ9ej7f5tEFqn4hVBw67g8q4BFnc626A8VYLbHwC6pARabNpu6ykbhBtsoke9l',
  'href': 'https://www.facebook.com/sharon.merklin/posts/pfbid0vGqUJ9ej7f5tEFqn4hVBw67g8q4BFnc626A8VYLbHwC6pARabNpu6ykbhBtsoke9l'},
 {'label': 'Title', 'value': ''},
 {'label': 'Name', 'value': 'Sharon Merklin'}]

We can't assume that label_values[3] always means Name.

In [17]:
facebook_data[0]["label_values"][3]["value"]

'Christine Giver'

In [18]:
rows = []

for post in facebook_data:
    timestamp = post["timestamp"]
    reaction = None
    name = None

    for item in post["label_values"]:

        if item.get("label") == "Reaction":
            reaction = item.get("value")

        if item.get("label") == "Name":
            name = item.get("value")

    rows.append({
        "timestamp": timestamp,
        "reaction": reaction,
        "name": name
    })



The above code is cleaning by going through each reaction, looking through its label values, finding reaction, finding name, and making it one row

In [22]:
facebook_df = pd.DataFrame(rows)

facebook_df["timestamp"] = pd.to_datetime(
    facebook_df["timestamp"],
    unit="s"
)

facebook_df.head()

,timestamp,reaction,name
0,2026-08-31 00:03:45,Like,Christine Giver
1,2026-08-30 15:14:40,Love,Jessica Ryan
2,2026-08-30 12:14:01,Like,NaN
3,2026-08-30 12:13:44,Like,Karl Kosko
4,2026-08-30 10:53:54,Like,Carol Bonavita


Why don't all of them have a name? Sometimes there might be a group liking something, which would have a group and an author. 

In [20]:
facebook_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10756 entries, 0 to 10755
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   timestamp  10756 non-null  int64
 1   reaction   10678 non-null  str  
 2   name       8888 non-null   str  
dtypes: int64(1), str(2)
memory usage: 436.4 KB


In [23]:
facebook_df["reaction"].value_counts()

reaction
Like     8454
Love     2140
Haha       42
Sad        36
Angry       4
Wow         2
Name: count, dtype: int64

In [27]:
facebook_df.groupby("name")["timestamp"].count()

name
1000 Libraries                                1
13abc                                         3
330ToGO                                       2
97.1 The Fan                                  1
ABC News                                      1
                                           ... 
Zane Cottingim                                1
Zoe Lambert                                  11
cleveland.com                                 1
vikesverified                                 1
ÐÐ°ÑÐ° ÐÐºÐ°ÑÐµÑÐ¸Ð½Ð° ÐÐ¾ÑÐ¾Ð½Ð°    120
Name: timestamp, Length: 1275, dtype: int64

In [29]:
facebook_df.groupby("name")["timestamp"].count().sort_values(
    ascending=False
)

name
Katie Dolciato                              183
Angie Gubanc                                169
Samantha Drayer                             123
ÐÐ°ÑÐ° ÐÐºÐ°ÑÐµÑÐ¸Ð½Ð° ÐÐ¾ÑÐ¾Ð½Ð°    120
Midwest Robot Combat Association - MRCA     116
                                           ... 
Tracy Small                                   1
Tyler Smith                                   1
Alex Mills                                    1
Wendy Shreffler                               1
Washington Local Schools                      1
Name: timestamp, Length: 1275, dtype: int64